In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import pickle

In [ ]:

project = 'SurvSurfBenchmark_NCT00981058'
ds_name = 'real_NCT00981058'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
g_resol = 1
t_resol = 7
model_prefix = 'Joint'

DIR_MODEL_SAVE = os.path.join('sksurv_models', project, model_prefix)

if os.path.exists(DIR_MODEL_SAVE):
    shutil.rmtree(DIR_MODEL_SAVE)
    os.mkdir(DIR_MODEL_SAVE)
else:
    os.mkdir(DIR_MODEL_SAVE)

In [ ]:
from dataset_NCT00981058 import DatasetNCT00981058

ds_train = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='train', 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)
ds_val = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='val', 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)


In [ ]:
df_Xy_train = ds_train._get_df_Xy_trans_obs()
assert all(
    df_Xy_train.groupby(ds_train.colname_g)['subject'].apply(lambda x: x.value_counts().max() == 1)
)
df_Xy_train['event_observed'] = df_Xy_train['event_observed'].astype(bool)
df_Xy_train = df_Xy_train.sort_values('subject')

df_Xy_val = ds_val._get_df_Xy_trans_obs()
assert all(
    df_Xy_val.groupby(ds_train.colname_g)['subject'].apply(lambda x: x.value_counts().max() == 1)
)
df_Xy_val = df_Xy_val.sort_values('subject')
df_Xy_val['event_observed'] = df_Xy_val['event_observed'].astype(bool)

df_Xy_val_grid = ds_val._get_df_Xy_true_prob()
df_Xy_val_grid = df_Xy_val_grid.sort_values(['subject',ds_train.colname_g])
df_Xy_val_grid['event_observed'] = df_Xy_val_grid['event_observed'].astype(bool)


In [ ]:
df_Xy_train.head()

In [ ]:
cols_y = ['event_observed', 'duration']
cols_X = np.r_[[ds_train.colname_g], df_Xy_train.columns[df_Xy_train.columns.str.startswith('feat')]]

In [ ]:
cols_y

In [ ]:
cols_X

In [ ]:
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
def get_non_cox_model(model_class):
    def get_model(n_jobs, random_state):
        try:
            return model_class(n_jobs=n_jobs, random_state=random_state, min_samples_leaf=20, n_estimators=1000)
        except TypeError:
            return model_class(random_state=random_state, min_samples_leaf=20, n_estimators=50)
    return get_model

def get_cox_model(model_class):
    def get_model(n_jobs, random_state):
        try:
            return model_class(fit_baseline_model=True, n_jobs=n_jobs)
        except TypeError:
            try:
                return model_class(fit_baseline_model=True)
            except TypeError:
                return model_class()
    return get_model

model_name_to_class = {
    f'{model_prefix}_{GradientBoostingSurvivalAnalysis.__name__}':get_non_cox_model(GradientBoostingSurvivalAnalysis),
    f'{model_prefix}_{RandomSurvivalForest.__name__}':get_non_cox_model(RandomSurvivalForest),
    f'{model_prefix}_{CoxnetSurvivalAnalysis.__name__}':get_cox_model(CoxnetSurvivalAnalysis),
    f'{model_prefix}_{CoxPHSurvivalAnalysis.__name__}':get_cox_model(CoxPHSurvivalAnalysis),
}

In [ ]:
df_metrics = []
surv_funcs = dict()
event_to_model = dict()
seeds = [10,20,30,40,50]
for model_name, model_class in model_name_to_class.items():
    for seed in seeds:
        model_id = f'{model_name}_seed_{seed}'
        model = model_class(n_jobs=-1, random_state=seed)
        y = df_Xy_train[cols_y].to_records(index=False)
        X = df_Xy_train[cols_X]
        model.fit(X, y)

        model_path = os.path.join(DIR_MODEL_SAVE,f'{model_id}.pickle')
        with open(model_path, 'wb') as f:
            # Pickle the 'data' dictionary using the highest protocol available.
            pickle.dump(model, f)

        for g, df_train_sub in df_Xy_train.groupby(ds_train.colname_g):
            if all(df_train_sub['event_observed'] == 0):
                continue
            
            y = df_train_sub[cols_y].to_records(index=False)
            X = df_train_sub[cols_X]
            event_to_model[f'{model_id}_event_{g}'] = model
            record = {
                'model_name':model_name,
                'event':g,
                'seed':seed,
                'c_index_train':model.score(X,y)
            }

            df_val_sub = df_Xy_val.loc[
                df_Xy_val[ds_train.colname_g] == g,:
            ]
            y = df_val_sub[cols_y].to_records(index=False)
            X = df_val_sub[cols_X]
            record['c_index_val'] = model.score(X,y)
            df_metrics.append(record)

            df_val_sub = df_Xy_val_grid.loc[
                df_Xy_val_grid[ds_train.colname_g] == g,:
            ].drop(columns=['duration','event_observed']).drop_duplicates().sort_values('subject')
            X = df_val_sub[cols_X]

            surv_funcs[f'{model_id}_event_{g}'] = model.predict_survival_function(X=X)
        print(f'finished running model {model_name}')
df_metrics = pd.DataFrame(df_metrics)


In [ ]:
df_val_sub

In [ ]:
df_metrics.groupby(['model_name','event'])['c_index_val'].describe()

In [ ]:
df = df_metrics.copy()
df['val_as_frac_of_train'] = df_metrics['c_index_val']/df_metrics['c_index_train']
df.groupby(['model_name','event'])['val_as_frac_of_train'].describe()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(4, 3))
sns.boxplot(
    data=df_metrics,
    x='model_name',
    y='c_index_val',
    hue='event',
    ax=ax,
)
plt.xticks(rotation=20, ha='right')


In [ ]:
n_curves_per_model = [len(i) for i in surv_funcs.values()]

n_curves_per_model = n_curves_per_model[0]

In [ ]:
event_to_model.keys()

In [ ]:
df_is_event1_higher_prob = []
for model_name in model_name_to_class.keys():
    for seed in seeds:
        for subj, df in df_Xy_val.groupby('subject'):
            X = df[cols_X].iloc[[0],:].copy()
            X = pd.concat([X]*5)
            X['g_max_by_time'] = [1,2,3,4,5]
            curve_events = event_to_model[f'{model_name}_seed_{seed}_event_1.0']
            curve_events = curve_events.predict_survival_function(X=X)
            
            x_interp = np.arange(0, 250, 7)
            curve_events = np.array([c(x_interp) for c in curve_events])
            diff = np.diff(curve_events, axis=0)
            
            prop_t_event1_higher_prob = np.mean((diff < 0).any(axis=0))
            row = {
                'model_name':model_name,
                'seed': seed,
                'subj':subj,
                'prop_t_event1_higher_prob':prop_t_event1_higher_prob,
                'g':g
            }
            df_is_event1_higher_prob.append(row)
df_is_event1_higher_prob = pd.DataFrame(df_is_event1_higher_prob)


In [ ]:
sns.heatmap(curve_events)

In [ ]:
df_is_event1_higher_prob.head()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8, 3))
sns.boxplot(
    data=df_is_event1_higher_prob,
    x='model_name',
    y='prop_t_event1_higher_prob',
    hue='seed',
    ax=ax,
    showfliers=False
)
plt.xticks(rotation=20, ha='right')



In [ ]:
df_is_event1_higher_prob.groupby(['model_name','seed'])['prop_t_event1_higher_prob'].describe()

In [ ]:
df_attention = df_is_event1_higher_prob.loc[df_is_event1_higher_prob['prop_t_event1_higher_prob'] > 0.02,:]
df_attention

In [ ]:
df_attention['subj'].nunique()

In [ ]:
for model_name, seed, subj in df_attention.sort_values('prop_t_event1_higher_prob')[['model_name', 'seed','subj']].drop_duplicates().values[::10]:
    X = df_Xy_val.loc[df_Xy_val['subject'] == subj, cols_X].iloc[[0],:].copy()

    
    fig, ax = plt.subplots(1,1,figsize=(4,3))
    for g in [1., 2., 3., 4., 5.]:
        X['g_max_by_time'] = g
        curve_event = event_to_model[f'{model_name}_seed_{seed}_event_{g}'].predict_survival_function(X=X)[0]

        x_shared = curve_event.x
        y = curve_event(x_shared)

        ax.step(x_shared, y, label=g)
        ax.set(title=f'{model_name}\nseed {seed}, subj {subj}', xlabel='time', ylabel='S(t)')
        ax.legend()